# Yuda 3 — VLM Semi-Multimodal
4 pipeline quick test (500 samples/class, 2+3 epoch)

In [1]:
import sys
sys.path.insert(0, "../..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import open_clip
import timm

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.utils.load_data import load_train
from modules.models.factory import TrashClassifier
from modules.training.train import fit
from modules.data.dataset import TrashDataset, get_transforms

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"open_clip: {open_clip.__version__}")

/home/prayudahlah/projects/external/big-data-big-trouble/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
open_clip: 3.3.0


## Subsampled Dataset (500/class, 80/20)

In [2]:
df = load_train()
dfs = []
for label in ['0_Recyclable', '1_Electronic', '2_Organic']:
    dfs.append(df[df['label'] == label].iloc[:500])
df_sub = pd.concat(dfs).reset_index(drop=True)

from sklearn.model_selection import StratifiedShuffleSplit
_CLASS_TO_IDX = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
y = df_sub['label'].map(_CLASS_TO_IDX).values
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(df_sub, y))
df_train = df_sub.iloc[train_idx].reset_index(drop=True)
df_val = df_sub.iloc[val_idx].reset_index(drop=True)
print(f"Train: {len(df_train)} | Val: {len(df_val)}")
print(f"Train distribution: {df_train['label'].value_counts().to_dict()}")
print(f"Val distribution:   {df_val['label'].value_counts().to_dict()}")

Train: 1200 | Val: 300
Train distribution: {'0_Recyclable': 400, '2_Organic': 400, '1_Electronic': 400}
Val distribution:   {'1_Electronic': 100, '0_Recyclable': 100, '2_Organic': 100}


## Load CLIP Models & Transforms

In [3]:
CLIP_NAME = 'ViT-B-32'
CLIP_PRETRAINED = 'laion2b_s34b_b79k'

clip_model, _, clip_transform = open_clip.create_model_and_transforms(
    CLIP_NAME, pretrained=CLIP_PRETRAINED
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_NAME)
clip_model = clip_model.to(DEVICE)
clip_model.eval()

# Freeze text tower, only keep visual for fine-tuning
clip_visual = clip_model.visual
clip_dim = clip_visual.output_dim  # 512 for ViT-B/32
print(f"CLIP visual dim: {clip_dim}")
print(f"CLIP transform: {clip_transform}")

CLIP visual dim: 512
CLIP transform: Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeConvertMode()
    MaybeToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)


## Datasets
3 dataset variants: CLIP transforms, ConvNeXt transforms, and dual transforms

In [4]:
conv_train_tfm = get_transforms(augment=True, img_size=224)
conv_val_tfm = get_transforms(augment=False, img_size=224)

class SingleTransformDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform(img), _CLASS_TO_IDX[row['label']]

class DualTransformDataset(Dataset):
    def __init__(self, df, transform_a, transform_b):
        self.df = df
        self.transform_a = transform_a
        self.transform_b = transform_b
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform_a(img), self.transform_b(img), _CLASS_TO_IDX[row['label']]

def make_loader(ds, batch_size=16, shuffle=True):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=2, pin_memory=True)

# CLIP datasets
clip_train_ds = SingleTransformDataset(df_train, clip_transform)
clip_val_ds = SingleTransformDataset(df_val, clip_transform)

# ConvNeXt datasets
conv_train_ds = SingleTransformDataset(df_train, conv_train_tfm)
conv_val_ds = SingleTransformDataset(df_val, conv_val_tfm)

# Dual datasets
dual_train_ds = DualTransformDataset(df_train, clip_transform, conv_train_tfm)
dual_val_ds = DualTransformDataset(df_val, clip_transform, conv_val_tfm)

print(f"CLIP: {len(clip_train_ds)} train / {len(clip_val_ds)} val")
print(f"Conv: {len(conv_train_ds)} train / {len(conv_val_ds)} val")

CLIP: 1200 train / 300 val
Conv: 1200 train / 300 val


---
## Model Definitions

In [5]:
class CLIPClassifier(nn.Module):
    """CLIP visual encoder + classification head (compatible with fit())"""
    def __init__(self, clip_visual, num_features, num_classes=3):
        super().__init__()
        self.encoder = clip_visual
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))
    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True


class CLIPWithTextFeatures(nn.Module):
    """Option B: image features + text similarity scores"""
    def __init__(self, clip_model, num_classes=3):
        super().__init__()
        self.visual = clip_model.visual
        self.encoder = self.visual  # alias for quick_train
        self.logit_scale = clip_model.logit_scale
        dim = self.visual.output_dim

        # Pre-compute text embeddings for 3 class prompts
        prompts = ["recyclable waste", "electronic waste", "organic waste"]
        text_tokens = clip_tokenizer(prompts).to(DEVICE)
        with torch.no_grad():
            text_feat = clip_model.encode_text(text_tokens)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        self.register_buffer('text_features', text_feat)  # [3, 512]

        # Classifier: 512 (visual) + 3 (similarity) = 515
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(dim + 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        v = self.visual(x)
        v_norm = v / v.norm(dim=-1, keepdim=True)
        sim = v_norm @ self.text_features.T  # [B, 3]
        combined = torch.cat([v, sim * self.logit_scale.exp()], dim=1)
        return self.classifier(combined)
    def freeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = True


class MultiEncoderClassifier(nn.Module):
    """Option C: CLIP features + ConvNeXt features concatenated"""
    def __init__(self, clip_visual, clip_dim, convnext_name='convnext_tiny', num_classes=3):
        super().__init__()
        self.clip_encoder = clip_visual
        self.conv_encoder = timm.create_model(convnext_name, pretrained=True, num_classes=0)
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            self.conv_dim = self.conv_encoder(dummy).shape[-1]

        total_dim = clip_dim + self.conv_dim
        print(f"  CLIP dim: {clip_dim} | Conv dim: {self.conv_dim} | Total: {total_dim}")
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(total_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )
    def forward(self, clip_x, conv_x):
        clip_feat = self.clip_encoder(clip_x)
        conv_feat = self.conv_encoder(conv_x)
        combined = torch.cat([clip_feat, conv_feat], dim=1)
        return self.classifier(combined)
    def freeze_encoder(self):
        for p in self.clip_encoder.parameters():
            p.requires_grad = False
        for p in self.conv_encoder.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.clip_encoder.parameters():
            p.requires_grad = True
        for p in self.conv_encoder.parameters():
            p.requires_grad = True


results = []

---
## 0. Baseline: CLIP Zero-Shot

In [6]:
print("=" * 50)
print("Zero-Shot CLIP Baseline")
print("=" * 50)

prompts = ["recyclable waste", "electronic waste", "organic waste"]
text_tokens = clip_tokenizer(prompts).to(DEVICE)

with torch.no_grad():
    text_feat = clip_model.encode_text(text_tokens)
    text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

val_loader = make_loader(clip_val_ds, batch_size=32, shuffle=False)
all_preds, all_labels, all_conf = [], [], []

for images, labels in tqdm(val_loader, desc="Zero-shot"):
    images = images.to(DEVICE)
    with torch.no_grad():
        img_feat = clip_model.encode_image(images)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        similarity = (100.0 * img_feat @ text_feat.T).softmax(dim=-1)
        conf, preds = similarity.max(dim=1)
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.numpy())
    all_conf.extend(conf.cpu().numpy())

f1 = f1_score(all_labels, all_preds, average='macro')
f1_per = f1_score(all_labels, all_preds, average=None)
print(f"\nZero-shot Macro F1: {f1:.4f}")
print(f"Per-class F1: {f1_per}")
print(f"Avg confidence: {np.mean(all_conf):.4f}")
results.append({'name': 'A_zero_shot', 'best_val_f1': f1, 'f1_per_class': f1_per.tolist()})

Zero-Shot CLIP Baseline


Zero-shot: 100%|██████████| 10/10 [00:01<00:00,  7.08it/s]


Zero-shot Macro F1: 0.9533
Per-class F1: [0.93137255 1.         0.92857143]
Avg confidence: 0.9332


## Option A: CLIP + Confidence Filter
Gambar dengan confidence ≥ treshold langsung pakai prediksi CLIP, sisanya pakai ConvNeXt

In [7]:
print("=" * 50)
print("Option A: Confidence Filter (treshold=0.95)")
print("=" * 50)

# Load ConvNeXt for fallback
convnext = TrashClassifier('convnext_tiny', num_classes=3).to(DEVICE)
ckpt = torch.load(RESULTS / 'yuda_convnext_tiny.pt', map_location=DEVICE, weights_only=True)
convnext.load_state_dict(ckpt)
convnext.eval()

TRESHOLD = 0.95
val_loader = make_loader(clip_val_ds, batch_size=32, shuffle=False)
all_preds, all_labels = [], []
clip_only, conv_fallback = 0, 0

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Conf Filter"):
        images = images.to(DEVICE)
        img_feat = clip_model.encode_image(images)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        similarity = (100.0 * img_feat @ text_feat.T).softmax(dim=-1)
        conf, clip_preds = similarity.max(dim=1)

        high_conf = conf >= TRESHOLD
        clip_only += high_conf.sum().item()
        conv_fallback += (~high_conf).sum().item()

        # Fallsback ke ConvNeXt untuk low confidence
        # ConvNeXt needs different transforms — convert back
        final_preds = clip_preds.clone()
        low_idx = (~high_conf).nonzero(as_tuple=True)[0]
        if len(low_idx) > 0:
            # Re-run low conf images through ConvNeXt with its transforms
            low_images = images[low_idx]
            conv_out = convnext(low_images)
            final_preds[low_idx] = conv_out.argmax(dim=1)

        all_preds.extend(final_preds.cpu().numpy())
        all_labels.extend(labels.numpy())

f1 = f1_score(all_labels, all_preds, average='macro')
f1_per = f1_score(all_labels, all_preds, average=None)
print(f"CLIP confident: {clip_only} | ConvNeXt fallback: {conv_fallback}")
print(f"Macro F1: {f1:.4f}")
print(f"Per-class F1: {f1_per}")
results.append({'name': 'A_conf_filter', 'best_val_f1': f1, 'f1_per_class': f1_per.tolist()})

Option A: Confidence Filter (treshold=0.95)


Conf Filter: 100%|██████████| 10/10 [00:04<00:00,  2.09it/s]

CLIP confident: 217 | ConvNeXt fallback: 83
Macro F1: 0.9933
Per-class F1: [0.99 1.   0.99]


## Train Helper
Quick train loop untuk 2-phase (head + finetune, epoch kecil)

In [8]:
def quick_train(model, train_loader, val_loader, name, epochs_head=2, epochs_finetune=3):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    best_f1 = 0.0
    best_state = None

    # Phase 1: Head only
    print(f"  Phase 1 — Head ({epochs_head} epoch)")
    model.freeze_encoder()
    opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    for epoch in range(epochs_head):
        model.train()
        for x, y in tqdm(train_loader, desc=f"  Head E{epoch+1}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()
        # Val
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds.extend(model(x).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
        print(f"    E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")

    # Phase 2: Full finetune
    print(f"  Phase 2 — Full ({epochs_finetune} epoch)")
    model.unfreeze_encoder()
    model.load_state_dict(best_state)
    opt = torch.optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': 1e-4},
        {'params': model.classifier.parameters(), 'lr': 1e-3},
    ], weight_decay=1e-4)
    for epoch in range(epochs_finetune):
        model.train()
        for x, y in tqdm(train_loader, desc=f"  Full E{epoch+1}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()
        # Val
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds.extend(model(x).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
        print(f"    E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")

    # Restore best
    model.load_state_dict(best_state)
    return best_f1


def quick_train_dual(model, train_loader, val_loader, name, epochs_head=2, epochs_finetune=3):
    """Same as quick_train but for DualTransformDataset (returns clip_x, conv_x, y)"""
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    best_f1 = 0.0
    best_state = None

    print(f"  Phase 1 — Head ({epochs_head} epoch)")
    model.freeze_encoder()
    opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    for epoch in range(epochs_head):
        model.train()
        for clip_x, conv_x, y in tqdm(train_loader, desc=f"  Head E{epoch+1}", leave=False):
            clip_x, conv_x, y = clip_x.to(DEVICE), conv_x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(clip_x, conv_x), y)
            loss.backward()
            opt.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for clip_x, conv_x, y in val_loader:
                clip_x, conv_x, y = clip_x.to(DEVICE), conv_x.to(DEVICE), y.to(DEVICE)
                preds.extend(model(clip_x, conv_x).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
        print(f"    E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")

    print(f"  Phase 2 — Full ({epochs_finetune} epoch)")
    model.unfreeze_encoder()
    model.load_state_dict(best_state)
    opt = torch.optim.AdamW([
        {'params': model.clip_encoder.parameters(), 'lr': 1e-4},
        {'params': model.conv_encoder.parameters(), 'lr': 1e-4},
        {'params': model.classifier.parameters(), 'lr': 1e-3},
    ], weight_decay=1e-4)
    for epoch in range(epochs_finetune):
        model.train()
        for clip_x, conv_x, y in tqdm(train_loader, desc=f"  Full E{epoch+1}", leave=False):
            clip_x, conv_x, y = clip_x.to(DEVICE), conv_x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(clip_x, conv_x), y)
            loss.backward()
            opt.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for clip_x, conv_x, y in val_loader:
                clip_x, conv_x, y = clip_x.to(DEVICE), conv_x.to(DEVICE), y.to(DEVICE)
                preds.extend(model(clip_x, conv_x).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
        print(f"    E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")

    model.load_state_dict(best_state)
    return best_f1

## Option B: CLIP + Text Embedding Features
Concat visual features + text similarity scores

In [9]:
print("=" * 50)
print("Option B: Text Embedding Features")
print("=" * 50)

model_b = CLIPWithTextFeatures(clip_model, num_classes=3)
train_loader = make_loader(clip_train_ds, batch_size=16, shuffle=True)
val_loader = make_loader(clip_val_ds, batch_size=16, shuffle=False)
f1 = quick_train(model_b, train_loader, val_loader, 'B_text_features')
print(f"Option B Macro F1: {f1:.4f}")
results.append({'name': 'B_text_features', 'best_val_f1': f1, 'f1_per_class': []})

Option B: Text Embedding Features
  Phase 1 — Head (2 epoch)


    E1: val_f1=1.0000 (best=1.0000)


    E2: val_f1=0.9967 (best=1.0000)
  Phase 2 — Full (3 epoch)


    E1: val_f1=0.6962 (best=1.0000)


    E2: val_f1=0.7802 (best=1.0000)


    E3: val_f1=0.8104 (best=1.0000)
Option B Macro F1: 1.0000


## Option C: CLIP + ConvNeXt Multi-Encoder
Concat features dari 2 encoder berbeda

In [10]:
print("=" * 50)
print("Option C: Multi-Encoder (CLIP + ConvNeXt)")
print("=" * 50)

model_c = MultiEncoderClassifier(clip_visual, clip_dim, convnext_name='convnext_tiny')
train_loader = make_loader(dual_train_ds, batch_size=8, shuffle=True)  # half batch for 2 encoders
val_loader = make_loader(dual_val_ds, batch_size=8, shuffle=False)
f1 = quick_train_dual(model_c, train_loader, val_loader, 'C_multi_encoder')
print(f"Option C Macro F1: {f1:.4f}")
results.append({'name': 'C_multi_encoder', 'best_val_f1': f1, 'f1_per_class': []})

Option C: Multi-Encoder (CLIP + ConvNeXt)
  CLIP dim: 512 | Conv dim: 768 | Total: 1280
  Phase 1 — Head (2 epoch)


    E1: val_f1=0.9766 (best=0.9766)


    E2: val_f1=0.9833 (best=0.9833)
  Phase 2 — Full (3 epoch)


    E1: val_f1=0.9261 (best=0.9833)


    E2: val_f1=0.9701 (best=0.9833)


    E3: val_f1=0.8859 (best=0.9833)
Option C Macro F1: 0.9833


## Option D: Knowledge Distillation (CLIP → ConvNeXt)
Gunakan CLIP zero-shot probabilities sebagai soft labels untuk train ConvNeXt

In [12]:
print("=" * 50)
print("Option D: Knowledge Distillation")
print("=" * 50)

# Step 1: Generate soft labels from CLIP zero-shot
print("Step 1: Generating CLIP soft labels...")
train_loader = make_loader(clip_train_ds, batch_size=32, shuffle=False)
soft_labels = []
clip_model.eval()
with torch.no_grad():
    for images, _ in tqdm(train_loader, desc="Soft labels"):
        images = images.to(DEVICE)
        img_feat = clip_model.encode_image(images)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        probs = (100.0 * img_feat @ text_feat.T).softmax(dim=-1)
        soft_labels.append(probs.cpu())
soft_labels = torch.cat(soft_labels)  # [N, 3]

# Step 2: Create dataset with soft labels
class DistillDataset(Dataset):
    def __init__(self, df, soft_labels, transform):
        self.df = df
        self.soft_labels = soft_labels
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform(img), self.soft_labels[idx], _CLASS_TO_IDX[row['label']]

distill_train_ds = DistillDataset(df_train, soft_labels, conv_train_tfm)
distill_train_loader = make_loader(distill_train_ds, batch_size=16, shuffle=True)
# Val still uses ConvNeXt default (no soft labels needed)
distill_val_loader = make_loader(conv_val_ds, batch_size=16, shuffle=False)

# Step 3: Train ConvNeXt with KD loss
print("Step 2: Training student with KD loss...")
student = TrashClassifier('convnext_tiny', num_classes=3).to(DEVICE)
opt = torch.optim.AdamW([
    {'params': student.encoder.parameters(), 'lr': 1e-4},
    {'params': student.classifier.parameters(), 'lr': 1e-3},
], weight_decay=1e-4)

best_f1 = 0.0
TEMP = 2.0
ALPHA = 0.7  # weight for KL div

for epoch in range(5):  # 5 epoch langsung full
    student.train()
    for x, soft_y, hard_y in tqdm(distill_train_loader, desc=f"KD E{epoch+1}", leave=False):
        x, soft_y, hard_y = x.to(DEVICE), soft_y.to(DEVICE), hard_y.to(DEVICE)
        opt.zero_grad()
        logits = student(x)
        kd_loss = F.kl_div(
            F.log_softmax(logits / TEMP, dim=1),
            soft_y,
            reduction='batchmean',
        ) * (TEMP ** 2)
        ce_loss = F.cross_entropy(logits, hard_y)
        loss = ALPHA * kd_loss + (1 - ALPHA) * ce_loss
        loss.backward()
        opt.step()

    # Val
    student.eval()
    preds, labs = [], []
    with torch.no_grad():
        for x, y in distill_val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds.extend(student(x).argmax(dim=1).cpu().numpy())
            labs.extend(y.cpu().numpy())
    f1 = f1_score(labs, preds, average='macro')
    if f1 > best_f1:
        best_f1 = f1
    print(f"    E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")

print(f"Option D Macro F1: {best_f1:.4f}")
results.append({'name': 'D_distillation', 'best_val_f1': best_f1, 'f1_per_class': []})

Option D: Knowledge Distillation
Step 1: Generating CLIP soft labels...


Soft labels: 100%|██████████| 38/38 [00:05<00:00,  6.49it/s]


Step 2: Training student with KD loss...


    E1: val_f1=0.4654 (best=0.4654)


    E2: val_f1=0.4897 (best=0.4897)


    E3: val_f1=0.4619 (best=0.4897)


    E4: val_f1=0.4579 (best=0.4897)


    E5: val_f1=0.4672 (best=0.4897)
Option D Macro F1: 0.4897


---
## Summary

In [13]:
summary = pd.DataFrame(results)
summary = summary.sort_values('best_val_f1', ascending=False)
summary.to_csv(RESULTS / 'yuda3_vlm_summary.csv', index=False)
summary

,name,best_val_f1,f1_per_class
2,B_text_features,1.000000,[]
1,A_conf_filter,0.993333,"[0.99, 1.0, 0.99]"
3,C_multi_encoder,0.983264,[]
0,A_zero_shot,0.953315,"[0.9313725490196079, 1.0, 0.9285714285714286]"
4,D_distillation,0.489746,[]
